# Week 07 · 天気ハッカソン / Weather Hackathon

チーム全員の氏名：
自分の担当：

天気予報を使って、誰かの「いつ・どうする？」を助けるNotebookを作ろう。2〜3人で作り、最後に全チームが実演する。

Build a notebook that helps someone decide when or what to do using a weather forecast. Work in teams of 2–3; every team gives a short demo at the end.

## AIの使い方 / Working with AI

自力チェックA・Bは各自で、AIチャット・AI補完・他人の入力なし。授業資料、自分のメモ、Python公式文書、通常の構文補完は使えます。制作の時間は、エラーの説明・調べ方・小さな提案にAIを使えます。自分で書いた条件とループを、AIやエージェントに一括で置き換えさせない。

Solo checks A and B: work individually without AI chat, AI completion or someone else typing. Course materials, your notes, official Python docs and ordinary syntax completion are allowed. During team building, AI may help explain errors or suggest a small change. Keep and understand the conditions and loops you write yourself.

各自でNotebookを保存。チームの完成品だけでなく、自分のチェックと説明を残します。

## 1. 予報を読み込む / Load the forecast

最初の2つのコードセルは準備用。そのまま実行できます。追加インストール・APIキーは不要です。ネット接続に失敗したら、同じフォルダのweather-response.jsonを使います。取得成功時はこのファイルを更新するので、提出するデータと結果をそろえられます。

Live API first; the saved JSON is the fallback. Keep the JSON beside this notebook. A successful fetch updates that same file.

Source: https://open-meteo.com/en/docs
Time: Asia/Tokyo; temp: °C; rain: precipitation probability (%).

In [ ]:
import json
from urllib.request import urlopen
from urllib.parse import urlencode
from datetime import datetime, timezone

latitude = 35.86
longitude = 139.97
url = "https://api.open-meteo.com/v1/forecast?" + urlencode({
    "latitude": latitude, "longitude": longitude,
    "hourly": "temperature_2m,precipitation_probability",
    "forecast_hours": 24, "timezone": "Asia/Tokyo"
})

try:
    with urlopen(url, timeout=10) as response:
        data = json.load(response)
    data["_source"] = {"url": url,
                       "retrieved_at": datetime.now(timezone.utc).isoformat()}
    with open("weather-response.json", "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)
    print("Live API / APIから取得")
except (OSError, ValueError):
    with open("weather-response.json", encoding="utf-8") as file:
        data = json.load(file)
    print("Saved JSON / 保存した予報を使用")

print("Retrieved:", data.get("_source", {}).get("retrieved_at", "see forecast dates"))

## 2. 辞書のリストへ / A list of dictionaries

time・temp・rainの3つの値を持つリストforecastを作ります。この準備セルもそのまま実行。

In [ ]:
forecast = []
hourly = data["hourly"]
for i in range(len(hourly["time"])):
    temp = hourly["temperature_2m"][i]
    rain = hourly["precipitation_probability"][i]
    if temp is not None and rain is not None:
        forecast.append({"time": hourly["time"][i],
                         "temp": temp, "rain": rain})
print(len(forecast), "hours / 時間分")
print("Forecast:", forecast[0]["time"], "to", forecast[-1]["time"])

## 自力チェックA · 最初の10分 / Solo check A · first 10 minutes

次の3件で、rainが30以下、tempが10以上のtimeを表示する。先に出力を予想してMarkdownに書き、次にforとifを自分で書いて確かめる。

予想（実行前）／なぜ？：

Write your predicted output before running. Then write your own for and if; print time when rain <= 30 and temp >= 10.

In [ ]:
check_hours = [
    {"time": "A", "temp": 10, "rain": 30},
    {"time": "B", "temp": 9, "rain": 0},
    {"time": "C", "temp": 18, "rain": 60}
]

In [ ]:
# AIなしで、自分のforとifを書く。 / Write your own for and if.


実際の結果／予想との違い／自分のifの説明：

Actual result / difference from prediction / explain your if:

## 3. 条件を、自分たちの目的へ / Make the rule your own

まず実行。次に、誰の何を助けるかを一文書き、max_rainやmin_temp、ifの条件、表示内容を変えよう。30は30％、10は10°C。値は自分たちの基準です。

Our user / 使う人：
Our question / 助ける判断：

In [ ]:
max_rain = 30
min_temp = 10
for hour in forecast:
    if hour["rain"] <= max_rain and hour["temp"] >= min_temp:
        print(hour["time"], hour["temp"], "°C", hour["rain"], "%")

## 4. 繰り返し使う / Let the user try again

showで表示、warmerで最低気温を1°C上げて再表示、quitで終了。入力待ちの間はセルが実行中です。終わるにはquit。設定を変えると結果が変わるか確かめ、使う人に合う操作へ変えよう。

In [ ]:
command = ""
while command != "quit":
    command = input("show / warmer / quit: ").strip().lower()
    if command == "warmer":
        min_temp += 1
    if command in ["show", "warmer"]:
        print("Minimum temperature:", min_temp, "°C")
        matches = 0
        for hour in forecast:
            if hour["rain"] <= max_rain and hour["temp"] >= min_temp:
                print(hour["time"], hour["temp"], "°C", hour["rain"], "%")
                matches += 1
        if matches == 0:
            print("No matching hours / 条件に合う時間はありません")

## 自力チェックB · 最後の15分 / Solo check B · last 15 minutes

教員がその場で指定する条件へ変更し、合う時間数を数えて表示する。AI・相談なしで各自が書く。条件、予想、変更したコード、結果と理由を残す。

Announced condition / 変更条件：
Prediction / 予想：

In [ ]:
# その場で指定された変更を、AIなしで書く。 / Code the announced change without AI.


Result / 結果：
Why / なぜ？：
The line I changed / 自分で変えた行：

## 5. 2通り試す / Two checks

設定1 → 予想 → 実際の結果：
設定2 → 予想 → 実際の結果：

条件に合う時間がない場合も試す（例：min_temp = 50）。quitで終了できるか確認。

## 発表 / Demo
1. 誰の何を助ける？
2. 条件を変えて動くところを見せる。
3. ifかwhileの一行を説明する。

## 宿題 / Homework
発表でもらった意見を使い、同じ道具を一つ改善する。各自のNotebookにチーム全員の氏名、自分の担当、改善点と確認2件を書く。自力チェックA・Bの元のコードを残し、改善を追記する。実行済み.ipynbとweather-response.jsonをUNIPAへ提出。

締切：2026-11-18 23:59 JST

## 出典・データ / Source and data
Open-Meteo: https://open-meteo.com/
API取得か保存JSONか、取得日時・予報の日付を記す。保存データを今の予報とは呼ばない。

## AIを使った場合 / If you used AI
何に使い、どの結果を自分で確認した？

## AIメモ / AI note

使わなかった場合は「使っていない」。使った場合は短く記録：
- 相談したこと：
- 採用・修正した小さな提案：
- 自分で確かめた入力と結果：

If you did not use AI, say so. Otherwise name the help, the small suggestion you used or changed, and a result you checked yourself.

## 自分が示すもの / My evidence
自力チェックA・B、自分で書いた条件とループ、発表で説明する一行。まだ動かなくても、自分で書いた部分と試した変更を残して提出する。

Keep your own two checks, the condition and loop you wrote, and a line you can explain in the demo. If unfinished, submit your own attempt and the changes you tried.